In [3]:
import pandas as pd

# 1.Load 

In [4]:
df1 = pd.read_csv("Fake.csv")
df2 = pd.read_csv("True.csv")

In [5]:
df1["Label"] = 0 # Assign 0 for fake news
df2["Label"] = 1 # Assign 1 for real news 
df = pd.concat([df1,df2])
df = df.sample(frac=1).reset_index(drop=True)
df.shape

(44898, 5)

In [6]:
df.head()

,title,text,subject,date,Label
0,"As Trump meets biotech CEOs, farm advisers fre...",CHICAGO (Reuters) - U.S. President-elect Donal...,politicsNews,"January 12, 2017",1
1,"EU agrees registration rules for drones, downl...",BRUSSELS (Reuters) - Drone owners in Europe wi...,worldnews,"November 30, 2017",1
2,Any U.S. military transgender ban could face m...,NEW YORK (Reuters) - President Donald Trump’s ...,politicsNews,"July 26, 2017",1
3,Trump says he is not looking to phase in corpo...,WASHINGTON (Reuters) - President Donald Trump ...,politicsNews,"October 31, 2017",1
4,U.S. journalist among 19 killed in South Sudan...,NAIROBI (Reuters) - A United States citizen wo...,worldnews,"August 26, 2017",1


In [7]:
df.isna().sum().sort_values(ascending=True)

title      0
text       0
subject    0
date       0
Label      0
dtype: int64

In [8]:
df.drop_duplicates(inplace=True)
df.shape

(44689, 5)

In [9]:
df["subject"].value_counts()

subject
politicsNews       11220
worldnews           9991
News                9050
politics            6838
left-news           4459
Government News     1570
US_News              783
Middle-east          778
Name: count, dtype: int64

In [10]:
df["Content"] =df["title"]+ " " +df["text"]

# 2 .Preprocessing 

In [11]:
import re

In [12]:
def preprocessing(text):
    # convet to lower
    text =str(text).lower()
    # remove url's
    text = re.sub(r"http\S+","",text)
    # remove punctuations
    text = re.sub(r"[^a-zA-Z0-9\s]","",text)
    # remove html
    text = re.sub(r"<.*?>","",text)
    return text 
df["Content"] = df["Content"].apply(preprocessing)

# 3.Feature Extraction 

In [13]:
import nltk
from nltk.tokenize import word_tokenize 
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
stop_words = set(stopwords.words("english"))

In [14]:
def sentiment_extraction(text):
    tokens = word_tokenize(text)
    filtered = [word for word in tokens if word not in stop_words]
    return " ".join(filtered)
df["Content"] = df["Content"].apply(sentiment_extraction)

In [15]:
new_df = df.drop(["title","text"],axis=1).copy()

In [16]:
new_df.head()

,subject,date,Label,Content
0,politicsNews,"January 12, 2017",1,trump meets biotech ceos farm advisers fret em...
1,worldnews,"November 30, 2017",1,eu agrees registration rules drones downloads ...
2,politicsNews,"July 26, 2017",1,us military transgender ban could face major l...
3,politicsNews,"October 31, 2017",1,trump says looking phase corporate tax cut was...
4,worldnews,"August 26, 2017",1,us journalist among 19 killed south sudan figh...


In [17]:
def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []
    tokens = word_tokenize(text)
    for token in tokens:
        stemm_tokens = ps.stem(token)
        stemmed_words.append(stemm_tokens)
    return " ".join(stemmed_words)
new_df["Content"] = new_df["Content"].apply(stemming)

## 5. TF-DTF vectorization 

In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [19]:
tf = TfidfVectorizer(
    max_features = 5000,
)
x = tf.fit_transform(new_df["Content"])

In [20]:
x

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 6072160 stored elements and shape (44689, 5000)>

In [21]:
y = new_df["Label"]
y.shape

(44689,)

## 6.Dataset & Dataloader

In [22]:
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import TensorDataset,DataLoader

In [23]:
x_train,x_test,y_train,y_test =train_test_split(x,y,test_size=0.2,random_state=42)

In [24]:
x_train.shape

(35751, 5000)

In [25]:
x_test.shape

(8938, 5000)

In [26]:
x_train = x_train.toarray()
x_test = x_test.toarray()

In [33]:
train_set = TensorDataset(
    torch.tensor(x_train,dtype=torch.float32),
    torch.tensor(y_train.values,dtype=torch.float32)
)
test_set = TensorDataset(
    torch.tensor(x_test,dtype=torch.float32),
    torch.tensor(y_test.values,dtype=torch.float32)
)

In [34]:
train_loader = DataLoader(train_set,batch_size=64,shuffle=True)
test_loader = DataLoader(test_set,batch_size=64,shuffle=True)

## 7. Build RNN 

In [35]:
import torch.nn as nn
import torch.optim as optim 

In [43]:
class RNN(nn.Module):
    def __init__(self,input_size,hidden_size=128,num_layers=1):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        # RNN layer 
        self.model = nn.RNN(input_size,hidden_size,num_layers,batch_first=True)
        # FC layer
        self.fc = nn.Linear(hidden_size,1)
    def forward(self,x):
        h0 = torch.zeros(self.num_layers,x.size(0),self.hidden_size)
        out, _ = self.model(x,h0)

        out = self.fc(out[:,-1,:])
        return out
        

In [44]:
input_size = x_train.shape[1]
model =RNN(input_size)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

# 8, Train RNN()

In [46]:
epochs = 10
for epoch in range(epochs):
    model.train()
    for xb,yb in train_loader:
        optimizer.zero_grad()
        xb = xb.unsqueeze(1)
        output = model(xb)
        output  =torch.sigmoid(output.squeeze())
        loss = criterion(output,yb)
        loss.backward()
        optimizer.step()
    print(f"epoch{epoch+1}/{epochs} ; loss {loss.item()}")

epoch1/10 ; loss 0.027587667107582092
epoch2/10 ; loss 0.009706681594252586
epoch3/10 ; loss 0.0014056352665647864
epoch4/10 ; loss 0.001898324815556407
epoch5/10 ; loss 0.006439764518290758
epoch6/10 ; loss 0.0004686240281444043
epoch7/10 ; loss 0.0013946888502687216
epoch8/10 ; loss 0.0001016431488096714
epoch9/10 ; loss 0.00024142551410477608
epoch10/10 ; loss 0.00014115839439909905


# 9. Model Evalution

In [70]:
# Evalution 
model.eval()
with torch.no_grad():
    correct_values = 0
    tot_val = 0
    for xb,yb in test_loader:
        xb = xb.unsqueeze(1)

        output = model(xb)
        predicted = (torch.sigmoid(output.squeeze())>0.5).float()

        tot_val += yb.size(0)
        correct_values += (predicted == yb).sum().item()
    print(f"Accuracy:{correct_values/tot_val*100:.3f}%")

Accuracy:99.318%


# 10.Model Testing 

In [87]:
from scipy.sparse import hstack

In [73]:
def predict_news(news):
    news = preprocessing(news)
    news = sentiment_extraction(news)
    news = stemming(news)

    vector = tf.transform([news]).toarray()
    xb = torch.tensor(vector, dtype=torch.float32).unsqueeze(1)

    model.eval()
    with torch.no_grad():
        output = model(xb).squeeze()
        predicted = (torch.sigmoid(output) > 0.5).float().item()

    print("Real News" if predicted == 1.0 else "Fake News")

In [74]:
predict_news("WASHINGTON (Reuters) - The head of a conservative Republican faction in the U.S. Congress, who voted this month for a huge expansion of the national debt to pay for tax cuts, called himself a â€œfiscal conservativeâ€ on Sunday and urged budget restraint in 2018. In keeping with a sharp pivot under way among Republicans, U.S. Representative Mark Meadows, speaking on CBSâ€™ â€œFace the Nation,â€ drew a hard line on federal spending, which lawmakers are bracing to do battle over in January. When they return from the holidays on Wednesday, lawmakers will begin trying to pass a federal budget in a fight likely to be linked to other issues, such as immigration policy, even as the November congressional election campaigns approach in which Republicans will seek to keep control of Congress. President Donald Trump and his Republicans want a big budget increase in military spending, while Democrats also want proportional increases for non-defense â€œdiscretionaryâ€ spending on programs that support education, scientific research, infrastructure, public health and environmental protection. â€œThe (Trump) administration has already been willing to say: â€˜Weâ€™re going to increase non-defense discretionary spending ... by about 7 percent,â€™â€ Meadows, chairman of the small but influential House Freedom Caucus, said on the program. â€œNow, Democrats are saying thatâ€™s not enough, we need to give the government a pay raise of 10 to 11 percent. For a fiscal conservative, I donâ€™t see where the rationale is. ... Eventually you run out of other peopleâ€™s money,â€ he said. Meadows was among Republicans who voted in late December for their partyâ€™s debt-financed tax overhaul, which is expected to balloon the federal budget deficit and add about $1.5 trillion over 10 years to the $20 trillion national debt. â€œItâ€™s interesting to hear Mark talk about fiscal responsibility,â€ Democratic U.S. Representative Joseph Crowley said on CBS. Crowley said the Republican tax bill would require the  United States to borrow $1.5 trillion, to be paid off by future generations, to finance tax cuts for corporations and the rich. â€œThis is one of the least ... fiscally responsible bills weâ€™ve ever seen passed in the history of the House of Representatives. I think weâ€™re going to be paying for this for many, many years to come,â€ Crowley said. Republicans insist the tax package, the biggest U.S. tax overhaul in more than 30 years,  will boost the economy and job growth. House Speaker Paul Ryan, who also supported the tax bill, recently went further than Meadows, making clear in a radio interview that welfare or â€œentitlement reform,â€ as the party often calls it, would be a top Republican priority in 2018. In Republican parlance, â€œentitlementâ€ programs mean food stamps, housing assistance, Medicare and Medicaid health insurance for the elderly, poor and disabled, as well as other programs created by Washington to assist the needy. Democrats seized on Ryanâ€™s early December remarks, saying they showed Republicans would try to pay for their tax overhaul by seeking spending cuts for social programs. But the goals of House Republicans may have to take a back seat to the Senate, where the votes of some Democrats will be needed to approve a budget and prevent a government shutdown. Democrats will use their leverage in the Senate, which Republicans narrowly control, to defend both discretionary non-defense programs and social spending, while tackling the issue of the â€œDreamers,â€ people brought illegally to the country as children. Trump in September put a March 2018 expiration date on the Deferred Action for Childhood Arrivals, or DACA, program, which protects the young immigrants from deportation and provides them with work permits. The president has said in recent Twitter messages he wants funding for his proposed Mexican border wall and other immigration law changes in exchange for agreeing to help the Dreamers. Representative Debbie Dingell told CBS she did not favor linking that issue to other policy objectives, such as wall funding. â€œWe need to do DACA clean,â€ she said.  On Wednesday, Trump aides will meet with congressional leaders to discuss those issues. That will be followed by a weekend of strategy sessions for Trump and Republican leaders on Jan. 6 and 7, the White House said. Trump was also scheduled to meet on Sunday with Florida Republican Governor Rick Scott, who wants more emergency aid. The House has passed an $81 billion aid package after hurricanes in Florida, Texas and Puerto Rico, and wildfires in California. The package far exceeded the $44 billion requested by the Trump administration. The Senate has not yet voted on the aid.")

Real News


In [75]:
predict_news("Donald Trump just signed the GOP tax scam into law. Of course, that meant that he invited all of his craven, cruel GOP sycophants down from their perches on Capitol Hill to celebrate in the Rose Garden at the White House. Now, that part is bad enough   celebrating tax cuts for a bunch of rich hedge fund managers and huge corporations at the expense of everyday Americans. Of course, Trump is beside himself with glee, as this represents his first major legislative win since he started squatting in the White House almost a year ago. Thanks to said glee, in true Trumpian style, he gave a free-wheeling address, and a most curious subject came up as Trump was thanking the goons from the Hill. Somehow, Trump veered away from tax cuts, and started talking about the Congressional baseball shooting that happened over the summer.In that shooting, Rep. Steve Scalise, who is also the House Majority Whip, was shot and almost lost his life. Thanks to this tragic and stunning act of political violence, Scalise had a long recovery; in fact he is still in physical therapy. But, of course, vain and looks-obsessed Trump decided that he would congratulate Scalise, not on his survival and on his miraculous recovery, but on the massive amount of weight Scalise lost while he was practically dying. And make no mistake   Scalise is VERY lucky to be alive. According to doctors, when he arrived at the hospital, Scalise was actually, quote, in  imminent risk of death.  Here is the quote, via Twitter:How stunningly tone deaf does one have to be to say something like that? I never thought I d say this about a Republican that I, by all reasonable accounts, absolutely loathe, but I feel sorry for him. I am sorry he got shot, and I am even sorrier that he now has to stand there and listen to that orange buffoon talk about him like that.I am sure that Scalise is a much tougher man than Trump, though. I am equally sure that he also knows that Trump is an international embarrassment and a crazy man who never should have been allowed anywhere near the White House.Featured image via Alex Wong/Getty Images")

Fake News


In [76]:
predict_news(" WATCH: Paul Ryan Just Told Us He Doesnâ€™t Care About Struggling Families Living In Blue States") # Error

Real News


In [77]:
predict_news("Factbox: Trump on Twitter (Dec 28) - Vanity Fair, Hillary Clinton")

Real News


In [82]:
predict_news(" SNL Hilariously Mocks Accused Child Molester Roy Moore For Losing AL Senate Race (VIDEO)")

Fake News


In [84]:
predict_news(" WATCH: Lindsey Graham Trashes Media For Portraying Trump As â€˜Kooky,â€™ Forgets His Own Words")

Fake News


In [86]:
predict_news("It almost seems like Donald Trump is trolling America at this point. In the beginning, when he tried to gaslight the country by insisting that the crowd at his inauguration was the biggest ever   or that it was even close to the last couple of inaugurations   we all kind of scratched our heads and wondered what kind of bullshit he was playing at. Then when he started appointing people to positions they had no business being in, we started to worry that this was going to go much worse than we had expected.After 11 months of Donald Trump pulling the rhetorical equivalent of whipping his dick out and slapping it on every table he gets near, I think it s time we address what s happening: Dude is a straight-up troll. He gets pleasure out of making other people uncomfortable or even seeing them in distress. He actively thinks up ways to piss off people he doesn t like.Let s set aside just for a moment the fact that that s the least presidential  behavior anyone s ever heard of   it s dangerous.His latest stunt is one of the grossest yet.Everyone is, by now, used to Trump not talking about things he doesn t want to talk about, and making a huge deal out of things that not many people care about. So it wasn t a huge surprise when the president didn t discuss the Sandy Hook shooting of 2012 on the fifth anniversary of that tragic event. What was a huge surprise was that he not only consciously decided not to invite the victims  families to the White House Christmas party this year   as they have been invited every year since the massacre took place, along with others who share those concerns.In each of the past 4 years, President Obama invited gun violence prevention activists, gun violence survivors (including the Sandy Hook families) and supportive lawmakers to his Christmas party. Zero gun lobbyists were in attendance. pic.twitter.com/QePW9FtbSh  Shannon Watts (@shannonrwatts) December 15, 2017The last sentence of that tweet is important, because that s exactly who Donald Trump did invite to the White House Christmas party. Instead of victims. On the anniversary day.Yesterday was the 5 year mark of the mass shooting at Sandy Hook School, which went unacknowledged by the President. On the same day, he hosted a White House Christmas party to which he invited @NRA CEO Wayne LaPierre. Here he is at the party with @DanPatrick. pic.twitter.com/mUbKCIWGxB  Shannon Watts (@shannonrwatts) December 15, 2017Wayne LaPierre is the man who, in response to the Sandy Hook massacre, finally issued a statement that blamed gun violence on music, movies, and video games, and culminated with perhaps the greatest bit of irony any man has ever unintentionally conceived of: Isn t fantasizing about killing people as a way to get your kicks really the filthiest form of pornography? Yes. Yes, it is, Wayne.Anyway, Happy Holidays Merry Christmas from Donald Trump, everyone!Featured image via Alex Wong/Getty Images")

Fake News
